# Base-model bake-off (S5.4)

Runs identical prompts through every candidate in configs/bakeoff.yaml and writes per-model summaries plus the comparison table. No fine-tuning happens here.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
import os
os.environ["HAVYN_DRIVE"] = env.drive_dir or "/content/havyn-artifacts"
os.environ["HAVYN_REPO"] = "/content/MCP-Packet-Tracer"
print("HAVYN_DRIVE =", os.environ["HAVYN_DRIVE"])

In [ ]:
!python havyn-stage5/scripts/build_bench_prompts.py --valid havyn-stage5/datasets/havyn-intents-v1/valid.jsonl --target 120 --out havyn-stage5/datasets/bench-prompts-v1.jsonl

**WARNING - THIS CELL STARTS BASE-MODEL BAKE-OFF.** downloads several GB and runs every candidate on 125 prompts

In [ ]:
from colab_ux import gate, dry_run, warn_expensive

gate("bakeoff", require_hf=True)
if dry_run():
    print("[DRY_RUN] BASE-MODEL BAKE-OFF not started")
else:
    warn_expensive("BASE-MODEL BAKE-OFF", "downloads several GB and runs every candidate on 125 prompts")
    from benchmark_harness import run_bakeoff
    table = run_bakeoff(
        config_path='havyn-stage5/configs/bakeoff.yaml',
        prompts='havyn-stage5/datasets/bench-prompts-v1.jsonl',
        out_dir=str(pathlib.Path(env.drive_dir) / 'bakeoff'),
        repeat_prompts=3,
    )
    print(table.read_text())